In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\drdee\Downloads\malazan_characters_stats.csv")
df.head(10)

,Character,Army,Strength,Intelligence,Magic_Power,Loyalty,Morale,Battles_Fought,Victories
0,Anomander Rake,Tiste Andii,98,95,100,90,95,72,68
1,Quick Ben,Bridgeburners,65,96,98,88,87,54,50
2,Kalam Mekhar,Bridgeburners,90,84,10,85,82,61,58
3,Whiskeyjack,Bridgeburners,88,89,40,94,90,65,60
4,Dujek Onearm,Malazan 2nd,83,80,15,92,86,71,63
5,Ganoes Paran,Malazan Host,70,85,70,89,80,57,51
6,Apsalar,Former Claw,78,88,55,83,84,45,42
7,Karsa Orlong,Teblor,100,72,5,65,78,80,76
8,Fiddler,Bridgeburners,74,82,40,91,88,64,57
9,Tattersail,2nd Army,60,91,94,87,83,46,41


### Task 1 — Light Data Preprocessing

In [3]:
# a) (a) Handle missing values using column means.
numeric_cols = df.select_dtypes(include="number").columns # .columns takes only columns with with numeric values
# selectinhg the 7 numeric columns and calculating average for each and replacing missing values
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
df.isna().sum()

Character         0
Army              0
Strength          0
Intelligence      0
Magic_Power       0
Loyalty           0
Morale            0
Battles_Fought    0
Victories         0
dtype: int64

#### There are no missing values in the columns

In [5]:
# b)Standardise text: strip spaces and apply .str.title().
#selecting columns with text
text_cols = df.select_dtypes(include="str").columns
#stripping the white space and Title casing the columns
df[text_cols].apply(lambda col: col.str.strip().str.title())
df[text_cols].head(10)

,Character,Army
0,Anomander Rake,Tiste Andii
1,Quick Ben,Bridgeburners
2,Kalam Mekhar,Bridgeburners
3,Whiskeyjack,Bridgeburners
4,Dujek Onearm,Malazan 2nd
5,Ganoes Paran,Malazan Host
6,Apsalar,Former Claw
7,Karsa Orlong,Teblor
8,Fiddler,Bridgeburners
9,Tattersail,2nd Army


### Removed the white space and changed the text to title case.
### I did not change Malazan 2nd as it will change the actual faction name

In [6]:
# checking for the standardising
for c in text_cols:
    stripped_matches = df[c].equals(df[c].str.strip()) # compares currnt columns against the stripped ones.
    # compares current column against df[c].str.title()
    title_matches = df[c].equals(df[c].str.title())
    print(f"{c}: no extra spaces = {stripped_matches}, correct title case = {title_matches}")


Character: no extra spaces = True, correct title case = True
Army: no extra spaces = True, correct title case = False


### I have done a  check to see if there are any aspaces or trails.
### I have checked for title case.
### There are no spaces ,trails and the text is in title case.

### Task 2 — Create NumPy Arrays 

In [7]:
# Converting Strength, Magic_Power, Morale, Battles_Fought, Victories into NumPy arrays
strength = df["Strength"].to_numpy()
magic_power = df["Magic_Power"].to_numpy()
morale = df["Morale"].to_numpy()
battles_fought = df["Battles_Fought"].to_numpy()
victories = df["Victories"].to_numpy()
# Printing the numoy arrays
print("Strength:", strength)
print("Magic_Power:", magic_power)
print("Morale:", morale)
print("Battles_Fought:", battles_fought)
print("Victories:", victories)



Strength: [ 98  65  90  88  83  70  78 100  74  60  68  55  99  85]
Magic_Power: [100  98  10  40  15  70  55   5  40  94  97  50  95  10]
Morale: [95 87 82 90 86 80 84 78 88 83 79 77 96 89]
Battles_Fought: [72 54 61 65 71 57 45 80 64 46 59 42 77 63]
Victories: [68 50 58 60 63 51 42 76 57 41 55 38 70 60]


### All the arrays are one dimensional arrays.

### Task 3 — Vectorised Computations

In [9]:
# Compute Win Rate (%) and Power Index using NumPy only
# Win Rate (%) = Victories / Battles_Fought * 100
win_rate = victories / battles_fought * 100

# Power Index = average of Strength, Magic_Power and Morale
power_index = (strength + magic_power + morale) / 3
print("Win Rate (%):", np.round(win_rate, 2))
print("Power Index:", np.round(power_index, 2))


Win Rate (%): [94.44 92.59 95.08 92.31 88.73 89.47 93.33 95.   89.06 89.13 93.22 90.48
 90.91 95.24]
Power Index: [97.67 83.33 60.67 72.67 61.33 73.33 72.33 61.   67.33 79.   81.33 60.67
 96.67 61.33]


### I rounded the  win_rate and power_index to 2 decimals using .round.
### The win rate ranges from 88.73 to 95.24 showing a high performance.
### The power index ranges from 60.67 to 97.67 shoeing wide spread.


### Task 4 — Correlation Analysis

In [10]:
# Use np.corrcoef to compute correlations.
# stacking 1D arrays to 2D matrix
corr_matrix = np.corrcoef([strength, magic_power, morale, win_rate, power_index])
labels = ["Strength", "Magic_Power", "Morale", "Win_Rate", "Power_Index"]

print("Correlation matrix (rows/cols:", labels, ")")
print(np.round(corr_matrix, 2)) # rounds all correlation coefficients to 2 decimals

Correlation matrix (rows/cols: ['Strength', 'Magic_Power', 'Morale', 'Win_Rate', 'Power_Index'] )
[[ 1.   -0.31  0.53  0.51  0.16]
 [-0.31  1.    0.26 -0.23  0.88]
 [ 0.53  0.26  1.    0.04  0.61]
 [ 0.51 -0.23  0.04  1.   -0.02]
 [ 0.16  0.88  0.61 -0.02  1.  ]]


### High positive correlation between magic power and power index.
### Moderate positive correlation between strength and win rate.
### Near zero correlation between win rate and power index 
### negative correlation between strength and magic power.

### Task 5 — Ranking Champions

In [11]:
# Sort characters by Power Index and display the top 3.
names = df["Character"].to_numpy()
order = np.argsort(power_index)[::-1]  # np.sort gives the sorted power scores as indices arranging in ascending order.
#[:-1} reverses the index array to descending order.
top3_idx = order[:3] # gives the top 3 values
# loops giving the ranks .
for rank, idx in enumerate(top3_idx, start=1):
    print(f"{rank}. {names[idx]} - Power Index: {power_index[idx]:.2f}")

Task 5 - Ranking Champions (Top 3 by Power Index)
1. Anomander Rake - Power Index: 97.67
2. Icarium - Power Index: 96.67
3. Quick Ben - Power Index: 83.33


In [12]:
# cleaned dataset
df.to_csv("cleaned_dataset.csv", index=False)